# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

> NOTE: DO NOT RUN THESE CELLS IF YOU ARE RUNNING THIS NOTEBOOK LOCALLY

In [ ]:
#!pip install -qU ragas==0.2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.7/175.7 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.1/71.1 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 68.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 411.6/411.6 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 454.8/454.8 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/1

In [ ]:
#!pip install -qU langchain-community==0.3.14 langchain-openai==0.2.14 unstructured==0.16.12 langgraph==0.2.61 langchain-qdrant==0.2.0

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

nltk.download('punkt') - Downloads the "punkt" tokenizer data. This is NLTK's sentence tokenizer that can split text into sentences and words. It's trained to recognize sentence boundaries even in complex cases like abbreviations, numbers, and punctuation.

nltk.download('averaged_perceptron_tagger') - Downloads the part-of-speech (POS) tagger data. This allows NLTK to identify the grammatical parts of speech (like nouns, verbs, adjectives, etc.) for each word in a sentence.

In [ ]:
import nltk
nltk.download('punkt') # python library for 
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/et/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/et/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [67]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Loan Data use-case!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [52]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders import TextLoader


path = "data/adn/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [53]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [54]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [55]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs[:20]:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 20, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [56]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/13 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/22 [00:00<?, ?it/s]

Property 'summary' already exists in node 'b3b322'. Skipping!
Property 'summary' already exists in node '698103'. Skipping!
Property 'summary' already exists in node 'bdfcbf'. Skipping!
Property 'summary' already exists in node 'c1ec05'. Skipping!
Property 'summary' already exists in node 'bbf724'. Skipping!
Property 'summary' already exists in node '5d0fcc'. Skipping!
Property 'summary' already exists in node 'd0d686'. Skipping!
Property 'summary' already exists in node '898f29'. Skipping!
Property 'summary' already exists in node '9d7a65'. Skipping!


Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/36 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'bdfcbf'. Skipping!
Property 'summary_embedding' already exists in node 'b3b322'. Skipping!
Property 'summary_embedding' already exists in node 'bbf724'. Skipping!
Property 'summary_embedding' already exists in node 'c1ec05'. Skipping!
Property 'summary_embedding' already exists in node '698103'. Skipping!
Property 'summary_embedding' already exists in node '9d7a65'. Skipping!
Property 'summary_embedding' already exists in node 'd0d686'. Skipping!
Property 'summary_embedding' already exists in node '898f29'. Skipping!
Property 'summary_embedding' already exists in node '5d0fcc'. Skipping!


Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 36, relationships: 245)

We can save and load our knowledge graphs as follows.

In [57]:
kg.save("loan_data_kg.json")
loan_data_kg = KnowledgeGraph.load("loan_data_kg.json")
loan_data_kg

KnowledgeGraph(nodes: 36, relationships: 245)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [58]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=loan_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [63]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

# query_distribution = [
#         (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 1.0),
# ]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.


Finally, we can use our `TestSetGenerator` to generate our testset!

In [65]:
testset = generator.generate(testset_size=10, query_distribution=query_distribution)
testset.to_pandas()

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

,user_input,reference_contexts,reference,synthesizer_name
0,What is the compendio de criterios juridico-la...,[COMPENDIO DE CRITERIOS JURÍDICO-LABORALES 199...,The compendio de criterios jurídico-laborales ...,single_hop_specifc_query_synthesizer
1,Cuáles son las causas de suspensión de contratos?,[Acoso Laboral . . . . . . . . . . . . . . . ....,Las causas de suspensión de contratos están li...,single_hop_specifc_query_synthesizer
2,¿Qué significa la huelga en el contexto del de...,[COMPENDIO DE CRITERIOS JURÍDICO-LABORALES 199...,La huelga es un término que aparece en el comp...,single_hop_specifc_query_synthesizer
3,What are the key labor legal criteria and regu...,[COMPENDIO DE CRITERIOS JURÍDICO-LABORALES 199...,The COMPENDIO DE CRITERIOS JURÍDICO-LABORALES ...,single_hop_specifc_query_synthesizer
4,What is COMPENDIO DE CRITERIOS JURÍDICO-LABORA...,[COMPENDIO DE CRITERIOS JURÍDICO-LABORALES 199...,COMPENDIO DE CRITERIOS JURÍDICO-LABORALES 1999...,single_hop_specifc_query_synthesizer
5,¿Cómo contribuyó la unificación de criterios j...,[<1-hop>\n\nCOMPENDIO DE CRITERIOS JURÍDICO-LA...,La unificación de criterios jurídicos laborale...,multi_hop_abstract_query_synthesizer
6,Cuales son los criterios juridico-laborales ac...,[<1-hop>\n\nCOMPENDIO DE CRITERIOS JURÍDICO-LA...,"Desde 2011, mediante la Directriz N° DMT-001-2...",multi_hop_abstract_query_synthesizer
7,¿Cómo contribuye la innovación en la gestión d...,[<1-hop>\n\nCOMPENDIO DE CRITERIOS JURÍDICO-LA...,"Desde 2011, mediante la Directriz N° DMT-001-2...",multi_hop_abstract_query_synthesizer
8,¿Cómo se relaciona la actualización del 'COMPE...,[<1-hop>\n\nCOMPENDIO DE CRITERIOS JURÍDICO-LA...,La actualización del 'COMPENDIO DE CRITERIOS J...,multi_hop_specific_query_synthesizer
9,"How do the labor criteria from 1999-2014, incl...",[<1-hop>\n\nCOMPENDIO DE CRITERIOS JURÍDICO-LA...,"The labor criteria from 1999-2014, as outlined...",multi_hop_specific_query_synthesizer


### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [62]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs[:20], testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/13 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/20 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/22 [00:00<?, ?it/s]

Property 'summary' already exists in node '511447'. Skipping!
Property 'summary' already exists in node 'f93a36'. Skipping!
Property 'summary' already exists in node '7e5146'. Skipping!
Property 'summary' already exists in node '8347e0'. Skipping!
Property 'summary' already exists in node 'bd9291'. Skipping!
Property 'summary' already exists in node 'e59831'. Skipping!
Property 'summary' already exists in node '04522b'. Skipping!
Property 'summary' already exists in node '4b084e'. Skipping!
Property 'summary' already exists in node '4a2b2f'. Skipping!


...

Property 'summary_embedding' already exists in node '7e5146'. Skipping!
Property 'summary_embedding' already exists in node 'f93a36'. Skipping!
Property 'summary_embedding' already exists in node '511447'. Skipping!
Property 'summary_embedding' already exists in node 'e59831'. Skipping!
Property 'summary_embedding' already exists in node '8347e0'. Skipping!
Property 'summary_embedding' already exists in node '04522b'. Skipping!
Property 'summary_embedding' already exists in node '4b084e'. Skipping!
Property 'summary_embedding' already exists in node '4a2b2f'. Skipping!
Property 'summary_embedding' already exists in node 'bd9291'. Skipping!


Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [69]:
testset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,What is the compendio de criterios juridico-la...,[COMPENDIO DE CRITERIOS JURÍDICO-LABORALES 199...,The compendio de criterios jurídico-laborales ...,single_hop_specifc_query_synthesizer
1,Cuáles son las causas de suspensión de contratos?,[Acoso Laboral . . . . . . . . . . . . . . . ....,Las causas de suspensión de contratos están li...,single_hop_specifc_query_synthesizer
2,¿Qué significa la huelga en el contexto del de...,[COMPENDIO DE CRITERIOS JURÍDICO-LABORALES 199...,La huelga es un término que aparece en el comp...,single_hop_specifc_query_synthesizer
3,What are the key labor legal criteria and regu...,[COMPENDIO DE CRITERIOS JURÍDICO-LABORALES 199...,The COMPENDIO DE CRITERIOS JURÍDICO-LABORALES ...,single_hop_specifc_query_synthesizer
4,What is COMPENDIO DE CRITERIOS JURÍDICO-LABORA...,[COMPENDIO DE CRITERIOS JURÍDICO-LABORALES 199...,COMPENDIO DE CRITERIOS JURÍDICO-LABORALES 1999...,single_hop_specifc_query_synthesizer
5,¿Cómo contribuyó la unificación de criterios j...,[<1-hop>\n\nCOMPENDIO DE CRITERIOS JURÍDICO-LA...,La unificación de criterios jurídicos laborale...,multi_hop_abstract_query_synthesizer
6,Cuales son los criterios juridico-laborales ac...,[<1-hop>\n\nCOMPENDIO DE CRITERIOS JURÍDICO-LA...,"Desde 2011, mediante la Directriz N° DMT-001-2...",multi_hop_abstract_query_synthesizer
7,¿Cómo contribuye la innovación en la gestión d...,[<1-hop>\n\nCOMPENDIO DE CRITERIOS JURÍDICO-LA...,"Desde 2011, mediante la Directriz N° DMT-001-2...",multi_hop_abstract_query_synthesizer
8,¿Cómo se relaciona la actualización del 'COMPE...,[<1-hop>\n\nCOMPENDIO DE CRITERIOS JURÍDICO-LA...,La actualización del 'COMPENDIO DE CRITERIOS J...,multi_hop_specific_query_synthesizer
9,"How do the labor criteria from 1999-2014, incl...",[<1-hop>\n\nCOMPENDIO DE CRITERIOS JURÍDICO-LA...,"The labor criteria from 1999-2014, as outlined...",multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [71]:
from langsmith import Client

client = Client()

dataset_name = "HR synthetic data"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="HR Synthetic Data"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [73]:
for data_row in testset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [74]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [78]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [76]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [79]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="HR RAG"
)

In [80]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [81]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\

Answer in spanish.

Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

For our LLM, we will be using TogetherAI's endpoints as well!

We're going to be using Meta Llama 3.1 70B Instruct Turbo - a powerful model which should get us powerful results!

In [82]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [83]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [88]:
rag_chain.invoke({"question" : "¿Puede mi patrono obligarme a marcar mis tiempos de descanso?"})

'De acuerdo con el contexto proporcionado, el hecho de que actualmente no haya que marcar la tarjeta al entrar y salir de los tiempos de descanso no implica un derecho adquirido o una condición más beneficiosa para el trabajador. Por lo tanto, es procedente que el patrono, en virtud de su poder de dirección, solicite a los trabajadores que marquen sus tiempos de descanso.\n\nEn resumen, sí, el patrono puede obligarte a marcar tus tiempos de descanso.'

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [85]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [86]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

empathy_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "empathy": "Is this response empathetic? Does it make the user feel like they are being heard?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`:
- `labeled_helpfulness_evaluator`:
- `empathy_evaluator`:

## LangSmith Evaluation

In [87]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'passionate-thunder-83' at:
https://smith.langchain.com/o/750ddb9f-5f75-492e-84ec-bd1afa0d60ec/datasets/6ad05793-be32-4e39-8635-78c878222b14/compare?selectedSessions=716faf7e-5f1d-4355-94f1-6772228fa325




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,What is the Compendio de Criterios Juridico-La...,El Compendio de Criterios Jurídico-Laborales 1...,None,The Compendio de Criterios Juridico-Laborales ...,1,1,0,4.600170,ba60a2dd-4b9c-4c0f-a93f-6a3cd19c2338,cc97534d-caed-41a5-9a50-0f93e8e3f0d5
1,"How do the labor criteria from 1999-2014, incl...",Los criterios jurídicos laborales del período ...,None,"The labor criteria from 1999-2014, as outlined...",1,1,0,4.319993,fec89f85-e9c8-4922-9379-124c95d19cd5,96c8fbf4-6dfa-49b5-a255-3cfd06d31359
2,¿Cómo se relaciona la actualización del 'COMPE...,"La actualización del ""COMPENDIO DE CRITERIOS J...",None,La actualización del 'COMPENDIO DE CRITERIOS J...,1,1,0,31.475910,60780594-b96e-4f2a-a253-6baaf87a9d3e,a92d7ade-2c9a-4982-918d-3bf3f1402108
3,¿Cómo contribuye la innovación en la gestión d...,"Desde 2011, mediante la Directriz N° DMT-001-2...",None,"Desde 2011, mediante la Directriz N° DMT-001-2...",1,1,0,7.476079,d9f575e8-9e2f-4ec5-b494-ee78808e8e63,a285446a-8801-491b-b135-1b792c4791c2
4,Cuales son los criterios juridico-laborales ac...,Los criterios jurídico-laborales actualizados ...,None,"Desde 2011, mediante la Directriz N° DMT-001-2...",1,1,0,6.711735,60ed0738-4cfa-44e4-aa75-a9dd2fde7a5f,8ad6f1e7-4098-4084-85e6-9708fe0bc88b
5,¿Cómo contribuyó la unificación de criterios j...,La unificación de criterios jurídicos laborale...,None,La unificación de criterios jurídicos laborale...,1,1,0,6.000390,cc6928c7-9fa0-404d-860e-a389e372f157,c039a0e4-e14b-4f9b-b7a7-8299ba83c9a5
6,What is COMPENDIO DE CRITERIOS JURÍDICO-LABORA...,"El ""COMPENDIO DE CRITERIOS JURÍDICO-LABORALES ...",None,COMPENDIO DE CRITERIOS JURÍDICO-LABORALES 1999...,0,1,0,2.889789,04df479f-f8a7-48ea-900f-ce430bdcf48c,8220c4a4-82e7-4a66-96c8-64452a1d780f
7,What are the key labor legal criteria and regu...,No se puede responder con la información propo...,None,The COMPENDIO DE CRITERIOS JURÍDICO-LABORALES ...,0,0,0,1.462415,a0f68958-befe-4dc0-8b1d-df4351786c30,96c2be15-5312-410a-8d17-9d56949ada1b
8,¿Qué significa la huelga en el contexto del de...,La huelga en el contexto del derecho laboral e...,None,La huelga es un término que aparece en el comp...,0,1,0,3.418131,f4876ed6-6702-49ca-b0cf-613de10ff1fd,ecd9c42e-a9e1-4030-9bb3-628ed13e65d1
9,Cuáles son las causas de suspensión de contratos?,No se especifican en el contexto completo las ...,None,Las causas de suspensión de contratos están li...,0,0,0,5.020395,1d81b273-ff0d-4cfb-9aad-082df3c41e9a,bf85c860-66a7-4970-afce-4f6f6f7f226b


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [89]:
EMPATHY_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

You must answer the question using empathy and kindness, and make sure the user feels heard.

Context: {context}
Question: {question}
"""

empathy_rag_prompt = ChatPromptTemplate.from_template(EMPATHY_RAG_PROMPT)

In [90]:
rag_documents = docs

In [91]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?

In [92]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?

In [94]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="HR Data for RAG"
)

In [95]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [96]:
empathy_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | empathy_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [98]:
empathy_rag_chain.invoke({"question" : "¿Puede mi patrono obligarme a marcar mis tiempos de descanso?"})

'Entiendo que te preocupa saber si tu patrono tiene el derecho de obligarte a marcar tus tiempos de descanso, y es muy importante que tengas claridad sobre este tema para proteger tus derechos y sentirte seguro en tu trabajo.\n\nSegún el contexto proporcionado, el patrono puede dictar medidas para ordenar el tiempo de descanso, incluso pudiendo impedir la salida de los trabajadores del lugar de trabajo durante este tiempo, con el fin de mantener la imagen de la empresa. Esto implica que sí puede establecer ciertas reglas sobre cómo se debe manejar el tiempo de descanso.\n\nAunque no se menciona explícitamente la obligación de "marcar" los tiempos de descanso, sí se indica que el patrono puede controlar y organizar este tiempo bajo sus órdenes. Por lo tanto, es posible que la práctica de marcar los descansos, como un control, sea una medida que el empleador implemente para supervisar este tiempo.\n\nEspero que esta información te ayude a entender mejor la situación. Si tienes más dudas 

Finally, we can evaluate the new chain on the same test set!

In [99]:
evaluate(
    empathy_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        empathy_evaluator
    ],
    metadata={"revision_id": "empathy_rag_chain"},
)

View the evaluation results for experiment: 'ample-way-43' at:
https://smith.langchain.com/o/750ddb9f-5f75-492e-84ec-bd1afa0d60ec/datasets/6ad05793-be32-4e39-8635-78c878222b14/compare?selectedSessions=66e0e625-1619-420f-902c-c9ddaf20fe41




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.empathy,execution_time,example_id,id
0,What is the Compendio de Criterios Juridico-La...,Thank you for your thoughtful question. Based ...,None,The Compendio de Criterios Juridico-Laborales ...,1,1,1,8.297495,ba60a2dd-4b9c-4c0f-a93f-6a3cd19c2338,d8774318-642d-4e60-bcff-e19b1fcaa91e
1,"How do the labor criteria from 1999-2014, incl...",Thank you for your thoughtful question. From t...,None,"The labor criteria from 1999-2014, as outlined...",1,1,1,5.823046,fec89f85-e9c8-4922-9379-124c95d19cd5,cba14349-2750-4807-9a7a-e0386bacd779
2,¿Cómo se relaciona la actualización del 'COMPE...,Entiendo que buscas comprender cómo la actuali...,None,La actualización del 'COMPENDIO DE CRITERIOS J...,1,1,1,5.754111,60780594-b96e-4f2a-a253-6baaf87a9d3e,52e5a6d5-0200-45e0-8a68-0102cec68c3e
3,¿Cómo contribuye la innovación en la gestión d...,Entiendo que quieres saber cómo la innovación ...,None,"Desde 2011, mediante la Directriz N° DMT-001-2...",1,1,1,3.947841,d9f575e8-9e2f-4ec5-b494-ee78808e8e63,25e5e27c-365b-442b-945f-1eaaf94b4914
4,Cuales son los criterios juridico-laborales ac...,Gracias por tu pregunta tan importante y valio...,None,"Desde 2011, mediante la Directriz N° DMT-001-2...",1,1,1,7.784862,60ed0738-4cfa-44e4-aa75-a9dd2fde7a5f,89528c62-afaf-4ff5-9fb5-fbfac6a7c42c
5,¿Cómo contribuyó la unificación de criterios j...,Gracias por tu pregunta tan importante y bien ...,None,La unificación de criterios jurídicos laborale...,1,1,1,10.637678,cc6928c7-9fa0-404d-860e-a389e372f157,3e2762e0-240d-4c5f-a28a-27d8d36b1d7a
6,What is COMPENDIO DE CRITERIOS JURÍDICO-LABORA...,Thank you for your question. Based on the cont...,None,COMPENDIO DE CRITERIOS JURÍDICO-LABORALES 1999...,0,1,1,3.889089,04df479f-f8a7-48ea-900f-ce430bdcf48c,b1daf55c-51f2-451c-bbda-429ddf8e2b5d
7,What are the key labor legal criteria and regu...,Thank you for your thoughtful question. Based ...,None,The COMPENDIO DE CRITERIOS JURÍDICO-LABORALES ...,1,0,1,10.621431,a0f68958-befe-4dc0-8b1d-df4351786c30,416bbda6-af05-412e-a2e3-558ca64d5bae
8,¿Qué significa la huelga en el contexto del de...,Entiendo que quieres comprender claramente qué...,None,La huelga es un término que aparece en el comp...,0,0,1,8.564807,f4876ed6-6702-49ca-b0cf-613de10ff1fd,e986f010-97fc-4069-9f23-c406eaa6da00
9,Cuáles son las causas de suspensión de contratos?,Entiendo que quieres conocer las causas de sus...,None,Las causas de suspensión de contratos están li...,0,0,1,4.622700,1d81b273-ff0d-4cfb-9aad-082df3c41e9a,60d9f6a6-73b1-4454-803d-69d1f638d66c


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.